# Semana 2 — El Mundo de la Aspiradora: Cuatro Arquitecturas de Agentes

**Plan de la lección:** [`../weeks/week-02.md`](../weeks/week-02.md) · **Diapositivas:** [`../slides/week-02/deck.md`](../slides/week-02/deck.md)

Programación en vivo, 25 minutos. Construye un entorno y luego ejecuta tres agentes usando una misma interfaz.

El orden es importante: cada agente falla de una manera que justifica la necesidad del siguiente.

**Idea clave que debes comprender:** el estado interno es lo que te permite *saber que has terminado*.

## 1 · El entorno

Dos habitaciones. Una percepción es `(ubicación, está_sucio_aquí)`. Observa lo que el agente

**no puede** ver: si la *otra* habitación está sucia.

In [1]:
# POO: Esta clase funciona como un molde para crear objetos que representan
# un mundo de aspiradora con dos habitaciones.
class VacuumWorld:
    """Two rooms. Percept = (location, status_here)."""

    # POO: __init__ es el constructor de la clase.
    # Se ejecuta automáticamente cada vez que creamos un objeto VacuumWorld.
    # Por defecto, las dos habitaciones comienzan sucias (True, True)
    # y la aspiradora empieza en la habitación 0.
    def __init__(self, dirt=(True, True), loc=0):

        # POO: self representa al objeto que se está creando o utilizando.
        # Estos son atributos del objeto, porque guardan su estado interno.
        #
        # self.dirt: guarda si cada habitación está sucia (True) o limpia (False).
        # self.loc: guarda la ubicación actual de la aspiradora (0 o 1).
        # self.score: guarda el puntaje acumulado.
        # self.t: guarda cuántos pasos han ocurrido.
        #
        # list(dirt) convierte la tupla recibida en una lista que luego podemos modificar.
        self.dirt, self.loc, self.score, self.t = list(dirt), loc, 0, 0

    # POO: percept() es un método de la clase.
    # Permite que la aspiradora observe el estado de la habitación donde está.
    def percept(self):

        # Devuelve una tupla:
        # (ubicación actual, estado de suciedad de esa habitación).
        #
        # Ejemplo:
        # (0, True) significa que está en la habitación 0 y está sucia.
        return (self.loc, self.dirt[self.loc])

    # POO: step() también es un método de la clase.
    # Recibe una acción y modifica el estado interno del objeto.
    def step(self, action):

        # Si la acción es "Suck", limpia la habitación donde se encuentra el agente.
        if action == "Suck":
            self.dirt[self.loc] = False

        # Si la acción es "Left", mueve la aspiradora a la habitación 0.
        elif action == "Left":
            self.loc = 0

        # Si la acción es "Right", mueve la aspiradora a la habitación 1.
        elif action == "Right":
            self.loc = 1

        # Performance measure:
        # después de ejecutar la acción, cuenta cuántas habitaciones están limpias.
        #
        # "not d" convierte:
        # True  (sucia)  -> False
        # False (limpia) -> True
        #
        # Como True vale 1 al sumar, cada habitación limpia aumenta el puntaje en 1.
        self.score += sum(not d for d in self.dirt)

        # Aumenta el tiempo en 1 porque se completó un paso de la simulación.
        self.t += 1

    # POO: __repr__ es un método especial.
    # Define cómo se mostrará el objeto cuando usamos print().
    def __repr__(self):

        # Crea una representación visual de las dos habitaciones.
        # "D" significa Dirty (sucia).
        # "." significa Clean (limpia).
        #
        # Ejemplo:
        # "DD" = las dos habitaciones están sucias.
        # ".D" = la habitación 0 está limpia y la 1 está sucia.
        rooms = "".join("D" if d else "." for d in self.dirt)

        # Devuelve un texto con:
        # estado de las habitaciones, ubicación del agente,
        # tiempo transcurrido y puntaje acumulado.
        return f"[{rooms}] agent@{self.loc} t={self.t} score={self.score}"


# POO: Aquí instanciamos la clase.
# Es decir, creamos un objeto llamado "w" a partir del molde VacuumWorld.
#
# Como no enviamos parámetros, usa los valores por defecto:
# dirt=(True, True) y loc=0.
w = VacuumWorld()

# Imprime primero la representación del objeto utilizando __repr__().
# Después llama al método percept() para mostrar lo que percibe la aspiradora.
print(w, "| percept:", w.percept())

[DD] agent@0 t=0 score=0 | percept: (0, True)


## 2 · Un agente reflejo simple

Relaciona directamente la percepción actual con una acción. No tiene ningún tipo de memoria.

In [2]:
# Agente reflejo simple: decide con lo que percibe ahora, sin recordar pasos anteriores.
# Es una función independiente, no un método: colabora con un objeto VacuumWorld.
def simple_reflex(percept):
    # Recibe la tupla del método percept(): ubicación y suciedad de la habitación actual.
    # loc y dirty son variables locales de esta función, no atributos de un objeto.
    loc, dirty = percept
    # Si dirty es True, hay suciedad y el agente elige aspirar.
    if dirty:
        # Retorna el texto "Suck" y termina la función; la limpieza se ejecuta después.
        return "Suck"
    # Si está limpio, retorna "Right" desde la habitación 0 o "Left" desde la 1.
    # El resultado siempre es una cadena que indica la acción elegida.
    return "Right" if loc == 0 else "Left"


# run coordina la simulación: recibe una función agente y retorna el objeto mundo al terminar.
# steps fija los pasos; dirt y loc, el estado inicial; trace, cuántos pasos mostrar.
def run(agent, steps=20, dirt=(True, True), loc=0, trace=0):
    # Instanciar una clase es crear un objeto: world es una instancia de VacuumWorld.
    # Sus atributos guardan el estado; sus métodos permiten consultarlo y modificarlo.
    world = VacuumWorld(dirt, loc)
    # Repite el ciclo de percibir, decidir y actuar durante steps pasos.
    for i in range(steps):
        # Llama al método del objeto para obtener la percepción y la entrega a agent.
        # Guarda en action la cadena retornada por el agente, por ejemplo "Suck".
        action = agent(world.percept()) #IMPORTANTE
        # Si se solicitó un recorrido, muestra los primeros trace pasos antes de actuar.
        if trace and i < trace:
            # La representación de world como texto usa su método especial __repr__.
            print(f"  {world}  ->  {action}")
        # El método step ejecuta la acción y actualiza los atributos del objeto.
        # Relación con POO: el agente decide y el objeto mundo administra su propio estado.
        world.step(action)
    # Retorna la instancia de VacuumWorld con el estado final, no solo su puntaje.
    return world


# Pasa simple_reflex como función, sin ejecutarla aquí; run la llama en cada paso.
# Ejecuta 8 pasos y los muestra; w referencia el objeto retornado por run.
w = run(simple_reflex, steps=8, trace=8)
# Muestra el objeto final usando la representación definida por __repr__.
print("\nfinal:", w)

  [DD] agent@0 t=0 score=0  ->  Suck
  [.D] agent@0 t=1 score=1  ->  Right
  [.D] agent@1 t=2 score=2  ->  Suck
  [..] agent@1 t=3 score=4  ->  Left
  [..] agent@0 t=4 score=6  ->  Right
  [..] agent@1 t=5 score=8  ->  Left
  [..] agent@0 t=6 score=10  ->  Right
  [..] agent@1 t=7 score=12  ->  Left

final: [..] agent@0 t=8 score=14


## 3 · Break the sensor — partial observability in four lines

Real sensors fail. Suppose the location sensor dies and reports `None`.

In [3]:
# BrokenSensorWorld hereda de VacuumWorld.
# En POO esto se llama HERENCIA:
# BrokenSensorWorld reutiliza los atributos y métodos de VacuumWorld,
# pero puede modificar alguno de sus comportamientos.
class BrokenSensorWorld(VacuumWorld):

    # Aquí sobrescribimos el método percept() que originalmente
    # pertenece a VacuumWorld.
    # En POO esto se conoce como sobrescritura de métodos (method overriding).
    def percept(self):

        # Retorna una tupla con dos valores:
        # (ubicación, estado de suciedad de la habitación actual)
        #
        # Pero aquí la ubicación se reemplaza por None,
        # simulando que el sensor de ubicación está dañado.
        #
        # self.dirt[self.loc] consulta si la habitación actual
        # está sucia (True) o limpia (False).
        return (None, self.dirt[self.loc])   # location unknown


try:
    # Se crea un OBJETO de la clase BrokenSensorWorld.
    # Aunque esta clase no tiene su propio __init__,
    # hereda el constructor de VacuumWorld.
    world = BrokenSensorWorld()

    # world.percept() llama al método percept() de BrokenSensorWorld.
    # Retorna algo como:
    # (None, True)
    #
    # None = no conoce su ubicación.
    # True = la habitación actual está sucia.
    #
    # Ese resultado se envía como argumento a simple_reflex().
    action = simple_reflex(world.percept())   # dirt=True, so it sucks — fine

    # simple_reflex() devuelve "Suck" porque detecta suciedad.
    # Luego world.step("Suck") modifica el estado interno del objeto:
    # la habitación actual pasa de sucia (True) a limpia (False).
    #
    # step() es un método heredado de VacuumWorld.
    world.step(action)

    # Muestra el estado actual del objeto y su percepción.
    # Después de aspirar, percept() debería retornar:
    # (None, False)
    print("after Suck:", world, "percept:", world.percept())

    # Se vuelve a llamar a percept().
    # Ahora retorna:
    # (None, False)
    #
    # La habitación está limpia, así que simple_reflex()
    # necesita decidir si moverse a la izquierda o derecha.
    # El problema es que la ubicación es None.
    action = simple_reflex(world.percept())   # now clean here; which way to move?

    # Si simple_reflex() logra retornar una acción,
    # se imprime la acción seleccionada.
    print("action chosen:", action)

# Si durante la ejecución ocurre un error,
# Python entra en este bloque.
except Exception as e:

    # type(e).__name__ retorna el nombre del tipo de error.
    # e contiene el mensaje del error.
    print(f"{type(e).__name__}: {e}")


# Este bloque simplemente imprime una explicación
# de lo que ocurrió en el experimento.
print("""
The agent cannot decide. With loc=None it has no basis for Left vs Right.
This is PARTIAL OBSERVABILITY, live, in four lines of change.
A reflex agent's competence is exactly as good as its sensors.
""")

after Suck: [.D] agent@0 t=1 score=1 percept: (None, False)
action chosen: Left

The agent cannot decide. With loc=None it has no basis for Left vs Right.
This is PARTIAL OBSERVABILITY, live, in four lines of change.
A reflex agent's competence is exactly as good as its sensors.



## 4 · Un agente basado en modelo — agregar memoria

La solución es usar un estado interno: registrar qué habitaciones han sido limpiadas y dónde creemos que estamos.

> **Nota — la creencia es tan buena como su suposición inicial.** Este agente
> *nunca lee la ubicación desde la percepción* (`_, dirty = percept`). Esto es
> intencional: es exactamente lo que le permite sobrevivir al sensor defectuoso anterior, donde
> la ubicación es `None`. El costo es que *asume que comienza en la habitación 0*. En un
> mundo donde el agente comienza en la habitación 1, limpiará esa habitación, creerá que
> ha terminado y dejará sucia la habitación 0 — esto se puede observar más abajo en su menor puntuación en la
> fila `loc=1`. La memoria te permite decir "sé que terminé"; pero no garantiza que tengas una
> creencia inicial correcta.

In [4]:
def model_based():
    # "state" es la memoria interna del agente.
    # loc guarda en qué habitación cree que está:
    # 0 = izquierda, 1 = derecha.
    #
    # clean guarda qué habitaciones ya sabe que están limpias.
    # [False, False] significa que todavía no sabe que ninguna está limpia.
    state = {"loc": 0, "clean": [False, False]}

    # Esta función representa al agente.
    # Recibe como entrada lo que percibe del ambiente.
    def agent(percept):

        # percept tiene la forma:
        # (ubicación, está_sucio)
        #
        # "_" significa que en esta parte no necesitamos usar
        # directamente la ubicación que viene en percept.
        _, dirty = percept

        # Si la habitación actual está sucia...
        if dirty:

            # El agente actualiza su memoria indicando que,
            # después de aspirar, esta habitación quedará limpia.
            state["clean"][state["loc"]] = True

            # Decide aspirar.
            return "Suck"

        # Si la habitación ya estaba limpia,
        # también la registra como limpia en su memoria.
        state["clean"][state["loc"]] = True

        # all() devuelve True si todos los valores de la lista son True.
        # Por ejemplo:
        # [True, True] -> True
        #
        # Si ambas habitaciones están limpias,
        # el agente sabe que ya terminó.
        if all(state["clean"]):
            return "NoOp"

        # Si todavía falta revisar otra habitación,
        # cambia su ubicación interna.
        #
        # Si estaba en 0:
        # 1 - 0 = 1
        #
        # Si estaba en 1:
        # 1 - 1 = 0
        state["loc"] = 1 - state["loc"]

        # Dependiendo de la nueva ubicación,
        # decide hacia dónde moverse.
        return "Right" if state["loc"] == 1 else "Left"

    # model_based() devuelve la función agent.
    # Esa función mantiene acceso a "state",
    # por eso puede recordar lo que ocurrió anteriormente.
    return agent


# model_based() crea el agente con memoria.
# run() ejecuta ese agente durante máximo 8 pasos.
w = run(model_based(), steps=8, trace=8)

# Mostramos el estado final del mundo.
print("\nfinal:", w)

  [DD] agent@0 t=0 score=0  ->  Suck
  [.D] agent@0 t=1 score=1  ->  Right
  [.D] agent@1 t=2 score=2  ->  Suck
  [..] agent@1 t=3 score=4  ->  NoOp
  [..] agent@1 t=4 score=6  ->  NoOp
  [..] agent@1 t=5 score=8  ->  NoOp
  [..] agent@1 t=6 score=10  ->  NoOp
  [..] agent@1 t=7 score=12  ->  NoOp

final: [..] agent@1 t=8 score=14


## 5 · La comparación

Ejecuta ambos agentes durante 20 pasos en cada configuración inicial y suma la puntuación total.

In [5]:
# Cada elemento de configs representa una configuración inicial del mundo.
# La primera parte indica qué habitaciones están sucias:
# True = sucia
# False = limpia
#
# La segunda parte indica dónde empieza la aspiradora:
# 0 = habitación izquierda
# 1 = habitación derecha

configs = [
    ((True, True), 0),     # Ambas sucias, empieza en la izquierda
    ((True, True), 1),     # Ambas sucias, empieza en la derecha
    ((True, False), 0),    # Izquierda sucia, derecha limpia
    ((False, True), 0),    # Izquierda limpia, derecha sucia
    ((False, False), 0)    # Ambas limpias
]


# Imprime los títulos de la tabla.
# <22 significa alinear el texto a la izquierda usando 22 espacios.
# >8 y >12 significan alinear los valores a la derecha.
print(f"{'start':<22} {'reflex':>8} {'model-based':>12}")

# Imprime una línea separadora de 44 caracteres.
print("-" * 44)


# Estas variables guardarán la suma total de los puntajes
# obtenidos por cada tipo de agente.
tot_r = tot_m = 0


# Recorremos cada configuración inicial guardada en configs.
#
# dirt recibe el estado de las habitaciones.
# loc recibe la ubicación inicial de la aspiradora.
for dirt, loc in configs:

    # Ejecuta el agente reflejo simple durante 20 pasos.
    # Al final obtiene únicamente su puntaje (.score).
    r = run(simple_reflex, 20, dirt, loc).score

    # Ejecuta el agente basado en modelo durante 20 pasos.
    # model_based() crea un agente nuevo con su propia memoria.
    # También obtenemos únicamente su puntaje final.
    m = run(model_based(), 20, dirt, loc).score

    # Acumulamos el puntaje del agente reflejo.
    tot_r += r

    # Acumulamos el puntaje del agente basado en modelo.
    tot_m += m

    # Creamos un texto para mostrar la configuración inicial.
    # Por ejemplo:
    # dirt=(True, True) loc=0
    label = f"dirt={dirt} loc={loc}"

    # Imprime una fila con:
    # configuración inicial | puntaje reflex | puntaje model-based
    print(f"{label:<22} {r:>8} {m:>12}")


# Imprime otra línea separadora.
print("-" * 44)


# Finalmente muestra la suma de todos los puntajes
# obtenidos por cada agente.
print(f"{'TOTAL':<22} {tot_r:>8} {tot_m:>12}")

start                    reflex  model-based
--------------------------------------------
dirt=(True, True) loc=0       38           38
dirt=(True, True) loc=1       38           20
dirt=(True, False) loc=0       40           40
dirt=(False, True) loc=0       39           39
dirt=(False, False) loc=0       40           40
--------------------------------------------
TOTAL                       195          177


### Qué nos dicen los números

El agente basado en modelo deja de moverse una vez que el mundo está limpio. El agente reflejo

oscila para siempre entre las habitaciones, y cada movimiento innecesario representa un costo que ya no puede recuperar.

> **El estado interno es lo que te permite saber que has terminado.**

Observa que esto no es simplemente una *regla* más inteligente. Es una *arquitectura* diferente.

## 6 · ⚠️ Obtienes lo que pides

Ahora se trata de un agente basado en la utilidad. El movimiento cuesta 1 punto; una habitación limpia aporta 1 punto por paso. Observa qué hace un agente *racional* con esa especificación.

In [6]:
# Crea un agente basado en utilidad.
# move_cost representa el costo de moverse de una habitación a otra.
def utility_agent(move_cost=1.0):

    # Estado interno o memoria del agente.
    # loc indica dónde está:
    # 0 = izquierda, 1 = derecha.
    # clean indica qué habitaciones sabe que están limpias.
    state = {"loc": 0, "clean": [False, False]}

    # Función interna que representa al agente.
    # Recibe como entrada lo que percibe del ambiente.
    def agent(percept):

        # percept tiene la forma (ubicación, está_sucio).
        # "_" indica que no se usa directamente la ubicación recibida.
        _, dirty = percept

        # Si la habitación actual está sucia...
        if dirty:

            # El agente registra en su memoria que,
            # después de aspirar, esa habitación quedará limpia.
            state["clean"][state["loc"]] = True

            # Decide aspirar.
            return "Suck"

        # Si la habitación ya estaba limpia,
        # también la guarda como limpia en su memoria.
        state["clean"][state["loc"]] = True

        # Calcula cuál es la otra habitación.
        # Si está en 0, other será 1.
        # Si está en 1, other será 0.
        other = 1 - state["loc"]

        # Compara si vale la pena cruzar a la otra habitación.
        # Solo se mueve si:
        # 1. todavía no sabe que la otra habitación está limpia
        # 2. el costo de moverse es menor que 1.0

        # Expected gain from crossing vs. the cost of crossing.
        if not state["clean"][other] and move_cost < 1.0:

            # Actualiza en su memoria la nueva ubicación.
            state["loc"] = other

            # Decide moverse a la derecha o izquierda.
            return "Right" if other == 1 else "Left"

        # Si moverse no vale la pena,
        # decide no hacer nada.
        return "NoOp"          # crossing is not worth it

    # Devuelve la función agent.
    return agent


# Prueba el agente con dos costos diferentes:
# primero 0.5 y luego 1.0.
for cost in (0.5, 1.0):

    # Ejecuta el agente durante 20 pasos.
    # El mundo empieza con:
    # habitación izquierda limpia
    # habitación derecha sucia
    # aspiradora en la habitación izquierda.
    w = run(utility_agent(cost), steps=20, dirt=(False, True), loc=0)

    # Muestra el resultado para cada costo.
    print(f"move_cost={cost}: {w}")


# Imprime una explicación del comportamiento observado.
print("""Con `move_cost=1.0`, el agente deja racionalmente la otra habitación sucia.
No está averiado; simplemente está maximizando la función de utilidad que escribiste.

Acabas de crear un agente que explota las especificaciones (*specification-gaming agent*). En la semana 14, esto recibe sus nombres técnicos: *reward hacking* (manipulación de la recompensa), ley de Goodhart o desalineación de indicadores (*proxy misalignment*).
Recuerda que ya habías creado uno en la semana 2, antes de conocer estos términos.
""")

move_cost=0.5: [..] agent@1 t=20 score=39
move_cost=1.0: [.D] agent@0 t=20 score=20
Con `move_cost=1.0`, el agente deja racionalmente la otra habitación sucia.
No está averiado; simplemente está maximizando la función de utilidad que escribiste.

Acabas de crear un agente que explota las especificaciones (*specification-gaming agent*). En la semana 14, esto recibe sus nombres técnicos: *reward hacking* (manipulación de la recompensa), ley de Goodhart o desalineación de indicadores (*proxy misalignment*).
Recuerda que ya habías creado uno en la semana 2, antes de conocer estos términos.



## 7 · Ejercicio para el estudio

1. Escribe un agente **basado en objetivos** que planifique una ruta en lugar de simplemente reaccionar.

2. Extiende `VacuumWorld` a *n* habitaciones. ¿Qué agentes siguen funcionando? ¿Cuáles necesitan
   ser reescritos? (Esta es la pregunta sobre representación atómica → factorizada de las
   diapositivas, aplicada ahora como una refactorización.)

3. Añade un 10 % de probabilidad de que `Suck` falle. ¿Qué arquitectura se degrada de manera más
   gradual y por qué?